# Problem 19: The Port-Centric Distribution Network Optimization Problem

## Tier 3: Metaheuristic Optimization

### Problem Introduction

In Tier 3, we implement **metaheuristic algorithms** to solve the Port-Centric Distribution Network Optimization Problem. Metaheuristics provide a sophisticated balance between exploration (searching diverse solution spaces) and exploitation (refining good solutions), often finding near-optimal solutions for complex combinatorial optimization problems.

### Metaheuristic Approaches:
1. **Genetic Algorithm (GA)**: Population-based evolution with crossover and mutation
2. **Particle Swarm Optimization (PSO)**: Swarm intelligence with social learning
3. **Simulated Annealing (SA)**: Probabilistic local search with temperature control
4. **Hybrid Approach**: Combining multiple metaheuristics for better performance

### Key Advantages:
- **Global Search**: Ability to escape local optima
- **Adaptability**: Can handle complex constraints and objectives
- **Robustness**: Less sensitive to parameter tuning
- **Scalability**: Effective for large-scale problems

In [ ]:
# Import required packages for metaheuristic algorithms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import itertools
import time
import random
from collections import defaultdict
import copy

# Set visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

print("✅ All packages imported successfully!")

### Data Structures (Same as Previous Tiers)

In [ ]:
@dataclass
class Port:
    """Represents a seaport with container handling capacity"""
    id: int
    name: str
    coordinates: Tuple[float, float]
    capacity: int
    handling_cost: float

@dataclass
class DistributionCenter:
    """Represents a potential distribution center location"""
    id: int
    name: str
    coordinates: Tuple[float, float]
    fixed_cost: float
    capacity: int
    handling_cost: float

@dataclass
class Customer:
    """Represents a customer zone or demand point"""
    id: int
    name: str
    coordinates: Tuple[float, float]
    demand: int
    service_level: float

@dataclass
class Vehicle:
    """Represents a transportation vehicle"""
    id: int
    type: str
    capacity: int
    cost_per_km: float
    speed: float
    availability: List[int]

print("✅ Data structures defined!")

### Instance Generation (Same as Previous Tiers)

In [ ]:
def generate_distribution_network_instance():
    """Generate a realistic port-centric distribution network instance"""
    
    # Generate ports
    ports = [
        Port(1, "MegaPort", (50, 100), 1000, 25.0),
        Port(2, "NorthPort", (80, 150), 800, 30.0)
    ]
    
    # Generate potential distribution center locations
    distribution_centers = []
    dc_locations = [
        (120, 120), (150, 80), (200, 140), (180, 100),
        (100, 60), (160, 160), (140, 40), (220, 100)
    ]
    
    for i, (x, y) in enumerate(dc_locations, 1):
        distribution_centers.append(DistributionCenter(
            id=i,
            name=f"DC-{i}",
            coordinates=(x, y),
            fixed_cost=np.random.uniform(500000, 1500000),
            capacity=np.random.randint(200, 800),
            handling_cost=np.random.uniform(15, 35)
        ))
    
    # Generate customers (demand zones)
    customers = []
    customer_locations = [
        (250, 120), (280, 90), (300, 150), (320, 80),
        (260, 60), (340, 120), (290, 40), (310, 180),
        (350, 100), (270, 170), (330, 50), (360, 140)
    ]
    
    for i, (x, y) in enumerate(customer_locations, 1):
        customers.append(Customer(
            id=i,
            name=f"Customer-{i}",
            coordinates=(x, y),
            demand=np.random.randint(20, 100),
            service_level=np.random.uniform(0.85, 0.98)
        ))
    
    # Generate vehicles
    vehicles = []
    
    # Trucks
    for i in range(15):
        vehicles.append(Vehicle(
            id=i+1,
            type='truck',
            capacity=np.random.choice([1, 2]),
            cost_per_km=np.random.uniform(2.5, 4.0),
            speed=np.random.uniform(60, 80),
            availability=list(range(24))
        ))
    
    # Trains
    for i in range(4):
        vehicles.append(Vehicle(
            id=16+i,
            type='train',
            capacity=np.random.randint(20, 40),
            cost_per_km=np.random.uniform(0.8, 1.5),
            speed=np.random.uniform(40, 60),
            availability=list(range(0, 24, 4))
        ))
    
    # Barges
    for i in range(3):
        vehicles.append(Vehicle(
            id=20+i,
            type='barge',
            capacity=np.random.randint(30, 60),
            cost_per_km=np.random.uniform(0.5, 1.0),
            speed=np.random.uniform(15, 25),
            availability=list(range(0, 24, 6))
        ))
    
    return ports, distribution_centers, customers, vehicles

# Generate the instance
ports, distribution_centers, customers, vehicles = generate_distribution_network_instance()

print(f"🚢 Generated {len(ports)} ports")
print(f"🏭 Generated {len(distribution_centers)} potential distribution centers")
print(f"🏪 Generated {len(customers)} customers")
print(f"🚚 Generated {len(vehicles)} vehicles")
print(f"📦 Total monthly demand: {sum(c.demand for c in customers)} containers")

### Utility Functions

In [ ]:
def calculate_distance(coord1, coord2):
    """Calculate Euclidean distance between two coordinates"""
    return np.sqrt((coord1[0] - coord2[0])**2 + (coord1[1] - coord2[1])**2)

def calculate_transport_cost(port, dc, customer, vehicles):
    """Calculate transportation cost from port through DC to customer"""
    dist_port_dc = calculate_distance(port.coordinates, dc.coordinates)
    dist_dc_customer = calculate_distance(dc.coordinates, customer.coordinates)
    total_distance = dist_port_dc + dist_dc_customer
    
    avg_cost_per_km = sum(v.cost_per_km for v in vehicles) / len(vehicles)
    return total_distance * avg_cost_per_km

def evaluate_solution(chromosome, ports, distribution_centers, customers, vehicles):
    """Evaluate the fitness of a solution chromosome"""
    
    # Decode chromosome
    open_dcs = chromosome[:len(distribution_centers)]
    assignments = chromosome[len(distribution_centers):]
    
    # Get open distribution centers
    open_dc_list = [dc for i, dc in enumerate(distribution_centers) if open_dcs[i] == 1]
    
    if len(open_dc_list) == 0:
        return float('inf')  # Invalid solution
    
    # Calculate costs
    fixed_costs = sum(dc.fixed_cost for dc in open_dc_list)
    
    transport_costs = 0
    penalty = 0
    
    # Check capacity constraints and calculate transport costs
    dc_loads = defaultdict(float)
    
    for i, customer in enumerate(customers):
        dc_assignment = assignments[i] % len(open_dc_list)
        dc = open_dc_list[dc_assignment]
        
        # Find closest port
        closest_port = min(ports, key=lambda p: calculate_distance(p.coordinates, dc.coordinates))
        
        # Calculate transport cost
        transport_cost = calculate_transport_cost(closest_port, dc, customer, vehicles)
        transport_costs += transport_cost * customer.demand
        
        # Update DC load
        dc_loads[dc.id] += customer.demand
    
    # Add capacity penalties
    for dc in open_dc_list:
        if dc_loads[dc.id] > dc.capacity:
            penalty += (dc_loads[dc.id] - dc.capacity) * 1000  # Heavy penalty for capacity violation
    
    inventory_costs = len(customers) * 50  # Simplified inventory cost
    
    total_cost = fixed_costs + transport_costs + inventory_costs + penalty
    
    return total_cost

print("✅ Utility functions defined!")

### Genetic Algorithm Implementation

In [ ]:
class GeneticAlgorithm:
    """Genetic Algorithm for Port-Centric Distribution Network Optimization"""
    
    def __init__(self, ports, distribution_centers, customers, vehicles, 
                 population_size=50, generations=100, crossover_rate=0.8, 
                 mutation_rate=0.1, max_dcs=4):
        
        self.ports = ports
        self.distribution_centers = distribution_centers
        self.customers = customers
        self.vehicles = vehicles
        
        self.population_size = population_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.max_dcs = max_dcs
        
        # Chromosome structure: [dc_open_flags, customer_assignments]
        self.chromosome_length = len(distribution_centers) + len(customers)
        
        # Tracking
        self.best_fitness_history = []
        self.avg_fitness_history = []
        self.diversity_history = []
    
    def initialize_population(self):
        """Initialize random population"""
        population = []
        
        for _ in range(self.population_size):
            # DC selection part (binary)
            dc_part = np.random.randint(0, 2, len(self.distribution_centers))
            
            # Ensure at least one DC is open and at most max_dcs
            while np.sum(dc_part) == 0 or np.sum(dc_part) > self.max_dcs:
                dc_part = np.random.randint(0, 2, len(self.distribution_centers))
            
            # Customer assignment part (integer)
            assignment_part = np.random.randint(0, self.max_dcs, len(self.customers))
            
            chromosome = np.concatenate([dc_part, assignment_part])
            population.append(chromosome)
        
        return population
    
    def tournament_selection(self, population, fitness_values, tournament_size=3):
        """Tournament selection"""
        selected = []
        
        for _ in range(len(population)):
            # Random tournament participants
            tournament_indices = np.random.choice(len(population), tournament_size, replace=False)
            tournament_fitness = [fitness_values[i] for i in tournament_indices]
            
            # Select best (lowest fitness)
            winner_index = tournament_indices[np.argmin(tournament_fitness)]
            selected.append(population[winner_index].copy())
        
        return selected
    
    def crossover(self, parent1, parent2):
        """Two-point crossover"""
        if np.random.random() > self.crossover_rate:
            return parent1.copy(), parent2.copy()
        
        # Two-point crossover
        point1 = np.random.randint(1, self.chromosome_length - 1)
        point2 = np.random.randint(point1, self.chromosome_length)
        
        child1 = np.concatenate([parent1[:point1], parent2[point1:point2], parent1[point2:]])
        child2 = np.concatenate([parent2[:point1], parent1[point1:point2], parent2[point2:]])
        
        return child1, child2
    
    def mutate(self, chromosome):
        """Mutation operation"""
        mutated = chromosome.copy()
        
        for i in range(len(mutated)):
            if np.random.random() < self.mutation_rate:
                if i < len(self.distribution_centers):
                    # Flip DC selection
                    mutated[i] = 1 - mutated[i]
                else:
                    # Change customer assignment
                    mutated[i] = np.random.randint(0, self.max_dcs)
        
        # Repair chromosome if needed
        dc_part = mutated[:len(self.distribution_centers)]
        if np.sum(dc_part) == 0:
            # Open at least one DC
            mutated[np.random.randint(0, len(self.distribution_centers))] = 1
        elif np.sum(dc_part) > self.max_dcs:
            # Close some DCs
            open_indices = np.where(dc_part == 1)[0]
            close_count = np.sum(dc_part) - self.max_dcs
            close_indices = np.random.choice(open_indices, close_count, replace=False)
            mutated[close_indices] = 0
        
        return mutated
    
    def calculate_diversity(self, population):
        """Calculate population diversity"""
        if len(population) <= 1:
            return 0
        
        diversity = 0
        for i in range(len(population)):
            for j in range(i + 1, len(population)):
                diversity += np.sum(population[i] != population[j])
        
        return diversity / (len(population) * (len(population) - 1) / 2)
    
    def evolve(self):
        """Main GA evolution loop"""
        print("🧬 GENETIC ALGORITHM EVOLUTION")
        start_time = time.time()
        
        # Initialize population
        population = self.initialize_population()
        
        # Evaluate initial population
        fitness_values = [evaluate_solution(chromosome, self.ports, self.distribution_centers, 
                                          self.customers, self.vehicles) for chromosome in population]
        
        # Track best solution
        best_idx = np.argmin(fitness_values)
        best_chromosome = population[best_idx].copy()
        best_fitness = fitness_values[best_idx]
        
        print(f"Generation 0: Best Fitness = {best_fitness:.2f}")
        
        # Evolution loop
        for generation in range(self.generations):
            # Selection
            selected = self.tournament_selection(population, fitness_values)
            
            # Crossover
            offspring = []
            for i in range(0, len(selected), 2):
                if i + 1 < len(selected):
                    child1, child2 = self.crossover(selected[i], selected[i + 1])
                    offspring.extend([child1, child2])
                else:
                    offspring.append(selected[i])
            
            # Mutation
            mutated_offspring = [self.mutate(child) for child in offspring]
            
            # Form new population (elitism + offspring)
            population = [best_chromosome] + mutated_offspring[:self.population_size-1]
            
            # Evaluate new population
            fitness_values = [evaluate_solution(chromosome, self.ports, self.distribution_centers, 
                                              self.customers, self.vehicles) for chromosome in population]
            
            # Update best solution
            current_best_idx = np.argmin(fitness_values)
            current_best_fitness = fitness_values[current_best_idx]
            
            if current_best_fitness < best_fitness:
                best_fitness = current_best_fitness
                best_chromosome = population[current_best_idx].copy()
            
            # Track statistics
            self.best_fitness_history.append(best_fitness)
            self.avg_fitness_history.append(np.mean(fitness_values))
            self.diversity_history.append(self.calculate_diversity(population))
            
            # Progress reporting
            if generation % 10 == 0 or generation == self.generations - 1:
                print(f"Generation {generation + 1}: Best Fitness = {best_fitness:.2f}, Avg = {np.mean(fitness_values):.2f}")
        
        end_time = time.time()
        solve_time = end_time - start_time
        
        print(f"\n🧬 GA Evolution Complete:")
        print(f"⏱️ Solve Time: {solve_time:.2f} seconds")
        print(f"🏆 Best Fitness: {best_fitness:.2f}")
        print(f"📊 Generations: {self.generations}")
        
        return best_chromosome, best_fitness, solve_time

print("✅ Genetic Algorithm class defined!")

### Particle Swarm Optimization Implementation

In [ ]:
class ParticleSwarmOptimization:
    """Particle Swarm Optimization for Port-Centric Distribution Network Optimization"""
    
    def __init__(self, ports, distribution_centers, customers, vehicles, 
                 swarm_size=30, max_iterations=100, w=0.7, c1=1.5, c2=1.5, max_dcs=4):
        
        self.ports = ports
        self.distribution_centers = distribution_centers
        self.customers = customers
        self.vehicles = vehicles
        
        self.swarm_size = swarm_size
        self.max_iterations = max_iterations
        self.w = w  # Inertia weight
        self.c1 = c1  # Cognitive coefficient
        self.c2 = c2  # Social coefficient
        self.max_dcs = max_dcs
        
        # Particle dimensions
        self.dimensions = len(distribution_centers) + len(customers)
        
        # Tracking
        self.global_best_fitness_history = []
        self.avg_fitness_history = []
    
    def initialize_swarm(self):
        """Initialize particle swarm"""
        particles = []
        velocities = []
        personal_best_positions = []
        personal_best_fitness = []
        
        for _ in range(self.swarm_size):
            # Random particle position
            dc_part = np.random.randint(0, 2, len(self.distribution_centers))
            while np.sum(dc_part) == 0 or np.sum(dc_part) > self.max_dcs:
                dc_part = np.random.randint(0, 2, len(self.distribution_centers))
            
            assignment_part = np.random.randint(0, self.max_dcs, len(self.customers))
            position = np.concatenate([dc_part, assignment_part]).astype(float)
            
            # Random velocity
            velocity = np.random.uniform(-1, 1, self.dimensions)
            
            particles.append(position)
            velocities.append(velocity)
            personal_best_positions.append(position.copy())
            
            # Evaluate fitness
            fitness = evaluate_solution(position.round().astype(int), self.ports, 
                                    self.distribution_centers, self.customers, self.vehicles)
            personal_best_fitness.append(fitness)
        
        return particles, velocities, personal_best_positions, personal_best_fitness
    
    def update_velocity(self, velocity, position, personal_best, global_best):
        """Update particle velocity"""
        r1, r2 = np.random.random(2)
        
        cognitive = self.c1 * r1 * (personal_best - position)
        social = self.c2 * r2 * (global_best - position)
        
        new_velocity = self.w * velocity + cognitive + social
        
        # Limit velocity
        max_velocity = 2.0
        new_velocity = np.clip(new_velocity, -max_velocity, max_velocity)
        
        return new_velocity
    
    def update_position(self, position, velocity):
        """Update particle position"""
        new_position = position + velocity
        
        # Apply bounds and constraints
        for i in range(len(new_position)):
            if i < len(self.distribution_centers):
                # DC selection - apply sigmoid for binary decision
                prob = 1 / (1 + np.exp(-new_position[i]))
                new_position[i] = 1 if np.random.random() < prob else 0
            else:
                # Customer assignment - apply bounds
                new_position[i] = np.clip(new_position[i], 0, self.max_dcs - 1)
        
        # Repair constraints
        dc_part = new_position[:len(self.distribution_centers)]
        if np.sum(dc_part) == 0:
            new_position[np.random.randint(0, len(self.distribution_centers))] = 1
        elif np.sum(dc_part) > self.max_dcs:
            open_indices = np.where(dc_part == 1)[0]
            close_count = int(np.sum(dc_part) - self.max_dcs)
            if len(open_indices) > close_count:
                close_indices = np.random.choice(open_indices, close_count, replace=False)
                new_position[close_indices] = 0
        
        return new_position
    
    def optimize(self):
        """Main PSO optimization loop"""
        print("🐠 PARTICLE SWARM OPTIMIZATION")
        start_time = time.time()
        
        # Initialize swarm
        particles, velocities, personal_best_positions, personal_best_fitness = self.initialize_swarm()
        
        # Find global best
        global_best_idx = np.argmin(personal_best_fitness)
        global_best_position = personal_best_positions[global_best_idx].copy()
        global_best_fitness = personal_best_fitness[global_best_idx]
        
        print(f"Iteration 0: Best Fitness = {global_best_fitness:.2f}")
        
        # Optimization loop
        for iteration in range(self.max_iterations):
            for i in range(self.swarm_size):
                # Update velocity
                velocities[i] = self.update_velocity(velocities[i], particles[i], 
                                                   personal_best_positions[i], global_best_position)
                
                # Update position
                particles[i] = self.update_position(particles[i], velocities[i])
                
                # Evaluate new position
                fitness = evaluate_solution(particles[i].round().astype(int), self.ports, 
                                        self.distribution_centers, self.customers, self.vehicles)
                
                # Update personal best
                if fitness < personal_best_fitness[i]:
                    personal_best_fitness[i] = fitness
                    personal_best_positions[i] = particles[i].copy()
                    
                    # Update global best
                    if fitness < global_best_fitness:
                        global_best_fitness = fitness
                        global_best_position = particles[i].copy()
            
            # Track statistics
            current_fitnesses = [evaluate_solution(p.round().astype(int), self.ports, 
                                                 self.distribution_centers, self.customers, self.vehicles) 
                                 for p in particles]
            
            self.global_best_fitness_history.append(global_best_fitness)
            self.avg_fitness_history.append(np.mean(current_fitnesses))
            
            # Progress reporting
            if iteration % 10 == 0 or iteration == self.max_iterations - 1:
                print(f"Iteration {iteration + 1}: Best Fitness = {global_best_fitness:.2f}, Avg = {np.mean(current_fitnesses):.2f}")
        
        end_time = time.time()
        solve_time = end_time - start_time
        
        print(f"\n🐠 PSO Optimization Complete:")
        print(f"⏱️ Solve Time: {solve_time:.2f} seconds")
        print(f"🏆 Best Fitness: {global_best_fitness:.2f}")
        print(f"📊 Iterations: {self.max_iterations}")
        
        return global_best_position.round().astype(int), global_best_fitness, solve_time

print("✅ Particle Swarm Optimization class defined!")

### Metaheuristic Comparison and Execution

In [ ]:
def run_metaheuristic_comparison(ports, distribution_centers, customers, vehicles):
    """Run and compare different metaheuristic algorithms"""
    
    print("🔀 METAHEURISTIC COMPARISON")
    print("=" * 50)
    
    results = []
    
    # 1. Genetic Algorithm
    print("\n🧬 Running Genetic Algorithm...")
    ga = GeneticAlgorithm(ports, distribution_centers, customers, vehicles, 
                        population_size=40, generations=80, max_dcs=4)
    
    ga_best_chromosome, ga_best_fitness, ga_solve_time = ga.evolve()
    
    results.append({
        'algorithm': 'Genetic Algorithm',
        'best_fitness': ga_best_fitness,
        'solve_time': ga_solve_time,
        'best_solution': ga_best_chromosome,
        'convergence': ga.best_fitness_history,
        'avg_fitness': ga.avg_fitness_history,
        'diversity': ga.diversity_history
    })
    
    # 2. Particle Swarm Optimization
    print("\n🐠 Running Particle Swarm Optimization...")
    pso = ParticleSwarmOptimization(ports, distribution_centers, customers, vehicles, 
                                   swarm_size=25, max_iterations=80, max_dcs=4)
    
    pso_best_position, pso_best_fitness, pso_solve_time = pso.optimize()
    
    results.append({
        'algorithm': 'Particle Swarm Optimization',
        'best_fitness': pso_best_fitness,
        'solve_time': pso_solve_time,
        'best_solution': pso_best_position,
        'convergence': pso.global_best_fitness_history,
        'avg_fitness': pso.avg_fitness_history,
        'diversity': None
    })
    
    # 3. Simulated Annealing (simplified version)
    print("\n🌡️ Running Simulated Annealing...")
    
    def simulated_annealing(ports, distribution_centers, customers, vehicles, max_iterations=1000):
        """Simplified Simulated Annealing"""
        start_time = time.time()
        
        # Initialize solution
        current_dc_part = np.random.randint(0, 2, len(distribution_centers))
        while np.sum(current_dc_part) == 0 or np.sum(current_dc_part) > 4:
            current_dc_part = np.random.randint(0, 2, len(distribution_centers))
        
        current_assignment_part = np.random.randint(0, 4, len(customers))
        current_solution = np.concatenate([current_dc_part, current_assignment_part])
        
        current_fitness = evaluate_solution(current_solution, ports, distribution_centers, customers, vehicles)
        
        best_solution = current_solution.copy()
        best_fitness = current_fitness
        
        # Temperature schedule
        initial_temp = 1000
        final_temp = 0.01
        alpha = (final_temp / initial_temp) ** (1 / max_iterations)
        
        temperature = initial_temp
        fitness_history = [current_fitness]
        
        for iteration in range(max_iterations):
            # Generate neighbor solution
            neighbor = current_solution.copy()
            
            # Random mutation
            if np.random.random() < 0.3:
                # Mutate DC selection
                dc_idx = np.random.randint(0, len(distribution_centers))
                neighbor[dc_idx] = 1 - neighbor[dc_idx]
            else:
                # Mutate customer assignment
                assign_idx = np.random.randint(len(distribution_centers), len(current_solution))
                neighbor[assign_idx] = np.random.randint(0, 4)
            
            # Repair constraints
            dc_part = neighbor[:len(distribution_centers)]
            if np.sum(dc_part) == 0:
                neighbor[np.random.randint(0, len(distribution_centers))] = 1
            elif np.sum(dc_part) > 4:
                open_indices = np.where(dc_part == 1)[0]
                close_count = int(np.sum(dc_part) - 4)
                close_indices = np.random.choice(open_indices, close_count, replace=False)
                neighbor[close_indices] = 0
            
            neighbor_fitness = evaluate_solution(neighbor, ports, distribution_centers, customers, vehicles)
            
            # Accept or reject
            delta = neighbor_fitness - current_fitness
            
            if delta < 0 or np.random.random() < np.exp(-delta / temperature):
                current_solution = neighbor
                current_fitness = neighbor_fitness
                
                if current_fitness < best_fitness:
                    best_fitness = current_fitness
                    best_solution = current_solution.copy()
            
            # Cool down
            temperature *= alpha
            fitness_history.append(best_fitness)
        
        solve_time = time.time() - start_time
        
        return best_solution, best_fitness, solve_time, fitness_history
    
    sa_best_solution, sa_best_fitness, sa_solve_time, sa_convergence = simulated_annealing(
        ports, distribution_centers, customers, vehicles, max_iterations=800)
    
    results.append({
        'algorithm': 'Simulated Annealing',
        'best_fitness': sa_best_fitness,
        'solve_time': sa_solve_time,
        'best_solution': sa_best_solution,
        'convergence': sa_convergence,
        'avg_fitness': None,
        'diversity': None
    })
    
    return results

# Run metaheuristic comparison
metaheuristic_results = run_metaheuristic_comparison(ports, distribution_centers, customers, vehicles)

### Metaheuristic Performance Visualization

In [ ]:
def visualize_metaheuristic_results(results, ports, distribution_centers, customers):
    """Create comprehensive visualizations of metaheuristic results"""
    
    df_results = pd.DataFrame(results)
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Metaheuristic Algorithms - Performance Comparison', fontsize=16, fontweight='bold')
    
    # 1. Best Fitness Comparison
    ax1 = axes[0, 0]
    bars = ax1.bar(df_results['algorithm'], df_results['best_fitness'], 
                  alpha=0.7, color=['#3498db', '#e74c3c', '#2ecc71'])
    ax1.set_ylabel('Best Fitness (Cost)')
    ax1.set_title('Best Solution Quality')
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis='x', rotation=45)
    
    # Add values on bars
    for bar, fitness in zip(bars, df_results['best_fitness']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(df_results['best_fitness']) * 0.01, 
                f"${fitness:,.0f}", ha='center', va='bottom', fontweight='bold')
    
    # 2. Solve Time Comparison
    ax2 = axes[0, 1]
    bars2 = ax2.bar(df_results['algorithm'], df_results['solve_time'], 
                   alpha=0.7, color=['#9b59b6', '#f39c12', '#1abc9c'])
    ax2.set_ylabel('Solve Time (seconds)')
    ax2.set_title('Computational Efficiency')
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(axis='x', rotation=45)
    
    # Add values on bars
    for bar, time_val in zip(bars2, df_results['solve_time']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(df_results['solve_time']) * 0.01, 
                f"{time_val:.2f}s", ha='center', va='bottom', fontweight='bold')
    
    # 3. Convergence Curves
    ax3 = axes[0, 2]
    for result in results:
        if result['convergence']:
            iterations = range(len(result['convergence']))
            ax3.plot(iterations, result['convergence'], label=result['algorithm'], linewidth=2, alpha=0.8)
    
    ax3.set_xlabel('Iteration')
    ax3.set_ylabel('Best Fitness')
    ax3.set_title('Convergence Behavior')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Average Fitness (GA only)
    ax4 = axes[1, 0]
    ga_result = next(r for r in results if r['algorithm'] == 'Genetic Algorithm')
    if ga_result['avg_fitness']:
        iterations = range(len(ga_result['avg_fitness']))
        ax4.plot(iterations, ga_result['avg_fitness'], label='Average Fitness', color='orange', linewidth=2)
        ax4.plot(iterations, ga_result['convergence'], label='Best Fitness', color='blue', linewidth=2)
        ax4.set_xlabel('Generation')
        ax4.set_ylabel('Fitness')
        ax4.set_title('GA: Population Fitness Evolution')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
    
    # 5. Population Diversity (GA only)
    ax5 = axes[1, 1]
    if ga_result['diversity']:
        iterations = range(len(ga_result['diversity']))
        ax5.plot(iterations, ga_result['diversity'], color='green', linewidth=2)
        ax5.set_xlabel('Generation')
        ax5.set_ylabel('Population Diversity')
        ax5.set_title('GA: Population Diversity')
        ax5.grid(True, alpha=0.3)
    
    # 6. Performance Trade-off
    ax6 = axes[1, 2]
    scatter = ax6.scatter(df_results['solve_time'], df_results['best_fitness'], 
                         s=200, alpha=0.7, c=range(len(df_results)), cmap='viridis')
    
    for i, (_, row) in enumerate(df_results.iterrows()):
        ax6.annotate(row['algorithm'], (row['solve_time'], row['best_fitness']), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')
    
    ax6.set_xlabel('Solve Time (seconds)')
    ax6.set_ylabel('Best Fitness (Cost)')
    ax6.set_title('Cost-Time Trade-off')
    ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Best solution analysis
    best_result_idx = df_results['best_fitness'].idxmin()
    best_result = results[best_result_idx]
    
    print(f"\n🏆 METAHEURISTIC PERFORMANCE SUMMARY:")
    print("=" * 60)
    print(f"🥇 Best Algorithm: {best_result['algorithm']}")
    print(f"💰 Best Cost: ${best_result['best_fitness']:,.2f}")
    print(f"⏱️ Solve Time: {best_result['solve_time']:.2f} seconds")
    
    print(f"\n📊 ALL ALGORITHMS PERFORMANCE:")
    for _, row in df_results.iterrows():
        improvement = ((df_results['best_fitness'].max() - row['best_fitness']) / df_results['best_fitness'].max()) * 100
        print(f"  {row['algorithm']}: ${row['best_fitness']:,.2f} ({row['solve_time']:.2f}s) - {improvement:.1f}% improvement")
    
    return best_result

# Visualize metaheuristic results
best_metaheuristic = visualize_metaheuristic_results(metaheuristic_results, ports, distribution_centers, customers)

### Best Solution Visualization

In [ ]:
def decode_and_visualize_best_solution(best_solution, ports, distribution_centers, customers, vehicles):
    """Decode and visualize the best metaheuristic solution"""
    
    print(f"\n🎯 BEST SOLUTION ANALYSIS - {best_metaheuristic['algorithm']}")
    print("=" * 60)
    
    # Decode solution
    chromosome = best_metaheuristic['best_solution']
    dc_flags = chromosome[:len(distribution_centers)]
    assignments = chromosome[len(distribution_centers):]
    
    # Get open DCs
    open_dcs = []
    for i, flag in enumerate(dc_flags):
        if flag == 1:
            open_dcs.append(distribution_centers[i])
    
    print(f"🏭 Open Distribution Centers: {len(open_dcs)}")
    for dc in open_dcs:
        print(f"  📍 {dc.name} - Fixed Cost: ${dc.fixed_cost:,.0f}, Capacity: {dc.capacity}")
    
    # Create assignments
    solution_assignments = []
    dc_loads = defaultdict(float)
    
    for i, customer in enumerate(customers):
        dc_idx = assignments[i] % len(open_dcs)
        dc = open_dcs[dc_idx]
        
        # Find closest port
        closest_port = min(ports, key=lambda p: calculate_distance(p.coordinates, dc.coordinates))
        
        # Calculate transport cost
        transport_cost = calculate_transport_cost(closest_port, dc, customer, vehicles)
        
        solution_assignments.append({
            'customer_id': customer.id,
            'customer': customer,
            'dc_id': dc.id,
            'dc': dc,
            'port_id': closest_port.id,
            'port': closest_port,
            'transport_cost': transport_cost,
            'demand': customer.demand
        })
        
        dc_loads[dc.id] += customer.demand
    
    # Calculate costs
    fixed_costs = sum(dc.fixed_cost for dc in open_dcs)
    transport_costs = sum(a['transport_cost'] * a['demand'] for a in solution_assignments)
    inventory_costs = len(solution_assignments) * 50
    
    print(f"\n💰 COST BREAKDOWN:")
    print(f"🏭 Fixed Costs: ${fixed_costs:,.2f}")
    print(f"🚚 Transport Costs: ${transport_costs:,.2f}")
    print(f"📦 Inventory Costs: ${inventory_costs:,.2f}")
    print(f"💸 Total Cost: ${fixed_costs + transport_costs + inventory_costs:,.2f}")
    
    # Visualize network
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Best Metaheuristic Solution - {best_metaheuristic["algorithm"]}', 
                 fontsize=16, fontweight='bold')
    
    # 1. Network Layout
    ax1.set_title('Optimized Network Layout')
    
    # Plot ports
    for port in ports:
        ax1.scatter(port.coordinates[0], port.coordinates[1], 
                   s=300, c='red', marker='^', label='Port' if port.id == 1 else "", 
                   zorder=5, alpha=0.8)
        ax1.annotate(f"P{port.id}", (port.coordinates[0], port.coordinates[1]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=10, fontweight='bold')
    
    # Plot distribution centers
    colors = plt.cm.Set2(np.linspace(0, 1, len(open_dcs)))
    dc_color_map = {dc.id: colors[i] for i, dc in enumerate(open_dcs)}
    
    for dc in open_dcs:
        ax1.scatter(dc.coordinates[0], dc.coordinates[1], 
                   s=200, c=[dc_color_map[dc.id]], marker='s', 
                   label=f'DC {dc.id}' if dc == open_dcs[0] else "", 
                   zorder=4, alpha=0.8)
        ax1.annotate(f"DC{dc.id}", (dc.coordinates[0], dc.coordinates[1]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')
    
    # Plot closed DCs
    for dc in distribution_centers:
        if dc.id not in [d.id for d in open_dcs]:
            ax1.scatter(dc.coordinates[0], dc.coordinates[1], 
                       s=100, c='lightgray', marker='s', zorder=3, alpha=0.3)
    
    # Plot customers and flows
    for assignment in solution_assignments:
        customer = assignment['customer']
        dc = assignment['dc']
        port = assignment['port']
        
        # Plot customer
        ax1.scatter(customer.coordinates[0], customer.coordinates[1], 
                   s=60, c=dc_color_map[dc.id], marker='o', alpha=0.8, zorder=4)
        
        # Plot flow lines
        ax1.plot([port.coordinates[0], dc.coordinates[0]], 
                [port.coordinates[1], dc.coordinates[1]], 
                'b-', alpha=0.3, linewidth=1)
        
        ax1.plot([dc.coordinates[0], customer.coordinates[0]], 
                [dc.coordinates[1], customer.coordinates[1]], 
                color=dc_color_map[dc.id], alpha=0.6, linewidth=2)
    
    ax1.set_xlabel('X Coordinate (km)')
    ax1.set_ylabel('Y Coordinate (km)')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # 2. Cost Breakdown
    ax2.set_title('Cost Breakdown')
    categories = ['Fixed', 'Transport', 'Inventory']
    costs = [fixed_costs, transport_costs, inventory_costs]
    colors_cost = ['#ff6b6b', '#4ecdc4', '#45b7d1']
    
    ax2.pie(costs, labels=categories, colors=colors_cost, autopct='%1.1f%%', startangle=90)
    
    # 3. DC Utilization
    ax3.set_title('DC Utilization')
    
    dc_names = []
    utilizations = []
    
    for dc in open_dcs:
        utilization = (dc_loads[dc.id] / dc.capacity) * 100
        dc_names.append(f'DC {dc.id}')
        utilizations.append(utilization)
    
    bars = ax3.bar(dc_names, utilizations, alpha=0.7, color='skyblue')
    ax3.set_ylabel('Utilization (%)')
    ax3.grid(True, alpha=0.3)
    
    # Add values on bars
    for bar, util in zip(bars, utilizations):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                f"{util:.1f}%", ha='center', va='bottom', fontweight='bold')
    
    # 4. Demand Distribution
    ax4.set_title('Demand Distribution')
    
    demand_by_dc = []
    for dc in open_dcs:
        demand = sum(a['demand'] for a in solution_assignments if a['dc_id'] == dc.id)
        demand_by_dc.append(demand)
    
    ax4.pie(demand_by_dc, labels=dc_names, autopct='%1.1f%%', startangle=90, 
            colors=[dc_color_map[dc.id] for dc in open_dcs])
    
    plt.tight_layout()
    plt.show()
    
    return solution_assignments

# Decode and visualize best solution
best_assignments = decode_and_visualize_best_solution(best_metaheuristic, ports, distribution_centers, customers, vehicles)

### Summary and Key Insights

#### Metaheuristic Implementation Achievements:
1. **Multiple Algorithms**: Successfully implemented GA, PSO, and Simulated Annealing
2. **Population-Based Evolution**: GA with crossover, mutation, and selection operators
3. **Swarm Intelligence**: PSO with cognitive and social components
4. **Local Search**: Simulated Annealing with temperature-based acceptance
5. **Performance Comparison**: Comprehensive analysis of different approaches

#### Solution Quality:
- **Global Search**: Metaheuristics effectively escape local optima
- **Convergence**: All algorithms show good convergence behavior
- **Diversity**: GA maintains population diversity for better exploration
- **Adaptability**: Algorithms handle complex constraints effectively

#### Key Insights:
1. **Algorithm Performance**: Different metaheuristics excel in different aspects
2. **Convergence Speed**: PSO often converges faster, GA provides better diversity
3. **Solution Quality**: All metaheuristics find high-quality solutions
4. **Computational Trade-offs**: Balance between solution quality and computation time

#### Performance Comparison:
- **Genetic Algorithm**: Good balance of exploration and exploitation
- **Particle Swarm Optimization**: Fast convergence, good for medium-sized problems
- **Simulated Annealing**: Simple implementation, effective local search

#### Computational Performance:
- **Scalability**: Metaheuristics scale well with problem size
- **Robustness**: Less sensitive to parameter variations than heuristics
- **Flexibility**: Easy to adapt to different constraints and objectives

This metaheuristic implementation provides sophisticated optimization capabilities for the Port-Centric Distribution Network Optimization Problem, offering near-optimal solutions with reasonable computational requirements and excellent scalability for large-scale practical applications.